# Random Forest Training

## Import Libraries

In [16]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline

# Reuse the Preprocessing Components Developed in Notebook 4
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

## Configure the Training

In [17]:
import numpy as np
import pandas as pd

PREDICTION_HORIZON_MINUTES = 60
RANDOM_SEED = 42

# Split Proportions
TRAIN_PROPORTION = 0.70
VALIDATION_PROPORTION = 0.15
TEST_PROPORTION = 0.15

PURGE_GAP = pd.Timedelta(minutes=PREDICTION_HORIZON_MINUTES)

assert np.isclose(
    TRAIN_PROPORTION + VALIDATION_PROPORTION + TEST_PROPORTION,
    1.0,
), "Split Proportions must Total 100%"

print("Training Configuration Loaded")
print(f"Prediction Horizon: {PREDICTION_HORIZON_MINUTES} minutes")
print(f"Purging Gap: {PURGE_GAP}")

Training Configuration Loaded
Prediction Horizon: 60 minutes
Purging Gap: 0 days 01:00:00


## Load Engineered Features

In [18]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ENGINEERED_FEATURES_PATH = (
    PROJECT_ROOT
    / "data"
    / "generated"
    / "engineered_features.csv"
)

# Stop early if Notebook 3 has not Exported the dataset
if not ENGINEERED_FEATURES_PATH.exists():
    raise FileNotFoundError(
        f"Engineered dataset not found:\n"
        f"{ENGINEERED_FEATURES_PATH}\n\n"
        "Run the Export cell in Notebook 3 First"
    )

# Load Timestamps
feature_df = pd.read_csv(
    ENGINEERED_FEATURES_PATH,
    parse_dates=[
        "timestamp",
        "target_action_timestamp",
    ],
)

excluded_columns = {
    "timestamp",
    "device_id",
    "target_action",
    "target_action_timestamp",
}

feature_columns = [
    column
    for column in feature_df.columns
    if column not in excluded_columns
]

X = feature_df[feature_columns].copy()
y = feature_df["target_action"].copy()

# Verify that the Expected columns exist and the Dataset is usable
required_columns = {
    "timestamp",
    "device_id",
    "target_action",
    "target_action_timestamp",
}

missing_columns = required_columns.difference(feature_df.columns)

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {sorted(missing_columns)}"
    )

if feature_df.empty:
    raise ValueError("The engineered dataset is empty.")

if y.isna().any():
    raise ValueError("The target column contains missing labels.")

if X.empty or not feature_columns:
    raise ValueError("No model features were found.")

print("Engineered Dataset Loaded Successfully")
print(f"Dataset Shape: {feature_df.shape}")
print(f"Number of Model Features: {len(feature_columns)}")
print(f"Number of Labelled Examples: {len(y)}")

print("\nTarget class Distribution:")
print(y.value_counts())

print("\nFeature Columns:")
print(feature_columns)

Engineered Dataset Loaded Successfully
Dataset Shape: (34560, 62)
Number of Model Features: 58
Number of Labelled Examples: 34560

Target class Distribution:
target_action
NO_ACTION          30649
ACTIVATE_PUMP        867
ACTIVATE_FAN         860
DEACTIVATE_FAN       824
DEACTIVATE_PUMP      782
INSPECT_DEVICE       578
Name: count, dtype: int64

Feature Columns:
['device_category', 'zone', 'temperature_c', 'humidity_pct', 'moisture_pct', 'water_ph', 'water_ec_ms_cm', 'water_flow_l_min', 'device_health', 'battery_pct', 'temperature_c_lag_1', 'temperature_c_lag_2', 'temperature_c_lag_4', 'humidity_pct_lag_1', 'humidity_pct_lag_2', 'humidity_pct_lag_4', 'moisture_pct_lag_1', 'moisture_pct_lag_2', 'moisture_pct_lag_4', 'water_ph_lag_1', 'water_ph_lag_2', 'water_ph_lag_4', 'water_ec_ms_cm_lag_1', 'water_ec_ms_cm_lag_2', 'water_ec_ms_cm_lag_4', 'water_flow_l_min_lag_1', 'water_flow_l_min_lag_2', 'water_flow_l_min_lag_4', 'device_health_lag_1', 'device_health_lag_2', 'device_health_lag_4', '

## Create Training, Validation, and Test Splits

In [19]:
# Use distinct timestamps to determine temporal Boundaries
unique_timestamps = pd.DatetimeIndex(
    feature_df["timestamp"].dropna().unique()
).sort_values()

if len(unique_timestamps) < 10:
    raise ValueError(
        "Insufficient Distinct Timestamps for chronological Splitting"
    )

train_cutoff_index = int(
    len(unique_timestamps) * TRAIN_PROPORTION
)

validation_cutoff_index = int(
    len(unique_timestamps)
    * (TRAIN_PROPORTION + VALIDATION_PROPORTION)
)

train_cutoff = unique_timestamps[train_cutoff_index]
validation_cutoff = unique_timestamps[validation_cutoff_index]

timestamps = feature_df["timestamp"]

train_mask = timestamps < train_cutoff

validation_mask = (
    (timestamps >= train_cutoff + PURGE_GAP)
    & (timestamps < validation_cutoff)
)

test_mask = (
    timestamps >= validation_cutoff + PURGE_GAP
)

X_train = X.loc[train_mask].copy()
y_train = y.loc[train_mask].copy()

X_validation = X.loc[validation_mask].copy()
y_validation = y.loc[validation_mask].copy()

X_test = X.loc[test_mask].copy()
y_test = y.loc[test_mask].copy()

# Retain the Corresponding Timestamps for Temporal Validation
train_timestamps = timestamps.loc[train_mask]
validation_timestamps = timestamps.loc[validation_mask]
test_timestamps = timestamps.loc[test_mask]

print("Splits Created")
print(f"Training rows:   {len(X_train):,}")
print(f"Validation rows: {len(X_validation):,}")
print(f"Test rows:       {len(X_test):,}")

print("\nTemporal boundaries:")
print(f"Training Cutoff:    {train_cutoff}")
print(f"Validation Cutoff:  {validation_cutoff}")
print(f"Purging Gap:        {PURGE_GAP}")

Splits Created
Training rows:   24,180
Validation rows: 5,148
Test rows:       5,136

Temporal boundaries:
Training Cutoff:    2026-01-21 23:45:00+00:00
Validation Cutoff:  2026-01-26 12:00:00+00:00
Purging Gap:        0 days 01:00:00


## Validate the Splits

In [20]:
assert len(X_train) > 0, "Training split is Empty"
assert len(X_validation) > 0, "Validation split is Empty"
assert len(X_test) > 0, "Test split is Empty"

# Ensure each Feature partition remains aligned with its labels
assert X_train.index.equals(y_train.index)
assert X_validation.index.equals(y_validation.index)
assert X_test.index.equals(y_test.index)

# Ensure each partition contains Timestamps in Chronological Order
assert train_timestamps.max() < validation_timestamps.min()
assert validation_timestamps.max() < test_timestamps.min()

# Confirm that the purge gaps meet the configured Horizon
assert (
    validation_timestamps.min() - train_timestamps.max()
    >= PURGE_GAP
), "Training-validation purge gap is Insufficient.="

assert (
    test_timestamps.min() - validation_timestamps.max()
    >= PURGE_GAP
), "Validation-test purge gap is Insufficient"

# Check that all partitions use the same Feature columns
assert X_train.columns.equals(X_validation.columns)
assert X_train.columns.equals(X_test.columns)

# Check for missing target Labels
assert not y_train.isna().any()
assert not y_validation.isna().any()
assert not y_test.isna().any()

# Report the resulting Partition sizes and target Distributions
split_summary = pd.DataFrame({
    "partition": ["Training", "Validation", "Test"],
    "rows": [
        len(X_train),
        len(X_validation),
        len(X_test),
    ],
    "start_timestamp": [
        train_timestamps.min(),
        validation_timestamps.min(),
        test_timestamps.min(),
    ],
    "end_timestamp": [
        train_timestamps.max(),
        validation_timestamps.max(),
        test_timestamps.max(),
    ],
})

print("All split Validation checks Passed\n")
display(split_summary)

print("\nTraining Target Distribution:")
display(y_train.value_counts().rename("count").to_frame())

print("\nValidation Target Distribution:")
display(y_validation.value_counts().rename("count").to_frame())

print("\nTest Target Distribution:")
display(y_test.value_counts().rename("count").to_frame())

All split Validation checks Passed



,partition,rows,start_timestamp,end_timestamp
0,Training,24180,2026-01-01 00:00:00+00:00,2026-01-21 23:30:00+00:00
1,Validation,5148,2026-01-22 00:45:00+00:00,2026-01-26 11:45:00+00:00
2,Test,5136,2026-01-26 13:00:00+00:00,2026-01-30 23:45:00+00:00



Training Target Distribution:


,count
target_action,
NO_ACTION,21410
ACTIVATE_PUMP,617
ACTIVATE_FAN,606
DEACTIVATE_FAN,580
DEACTIVATE_PUMP,551
INSPECT_DEVICE,416



Validation Target Distribution:


,count
target_action,
NO_ACTION,4563
ACTIVATE_PUMP,143
DEACTIVATE_PUMP,122
ACTIVATE_FAN,120
DEACTIVATE_FAN,111
INSPECT_DEVICE,89



Test Target Distribution:


,count
target_action,
NO_ACTION,4595
ACTIVATE_FAN,132
DEACTIVATE_FAN,128
DEACTIVATE_PUMP,104
ACTIVATE_PUMP,104
INSPECT_DEVICE,73


## Train and Evaluate the Baseline

In [21]:
# Establish a benchmark using only the Training labels
baseline_model = DummyClassifier(strategy="most_frequent")

baseline_model.fit(X_train, y_train)

# Evaluate on Validation Data
baseline_predictions = baseline_model.predict(X_validation)

baseline_accuracy = accuracy_score(
    y_validation,
    baseline_predictions,
)

baseline_macro_f1 = f1_score(
    y_validation,
    baseline_predictions,
    average="macro",
    zero_division=0,
)

baseline_results = pd.DataFrame({
    "model": ["Majority-Class Baseline"],
    "accuracy": [baseline_accuracy],
    "macro_f1": [baseline_macro_f1],
})

display(baseline_results)

print(f"Baseline Accuracy: {baseline_accuracy:.4f}")
print(f"Baseline macro-F1: {baseline_macro_f1:.4f}")
print(
    "Baseline Prediction: "
    f"{baseline_model.classes_[0] if len(baseline_model.classes_) == 1 else baseline_model.predict(X_validation.iloc[[0]])[0]}"
)

,model,accuracy,macro_f1
0,Majority-Class Baseline,0.886364,0.156627


Baseline Accuracy: 0.8864
Baseline macro-F1: 0.1566
Baseline Prediction: NO_ACTION


## Validate Class Coverage

In [22]:
# Compare observed Classes with the complete intended action Vocabulary
EXPECTED_ACTIONS = [
    "ACTIVATE_PUMP",
    "DEACTIVATE_PUMP",
    "ACTIVATE_FAN",
    "DEACTIVATE_FAN",
    "INSPECT_DEVICE",
    "NO_ACTION",
]

training_classes = set(y_train.unique())
validation_classes = set(y_validation.unique())
test_classes = set(y_test.unique())

class_coverage = pd.DataFrame({
    "action": EXPECTED_ACTIONS,
    "training_count": [
        int((y_train == action).sum())
        for action in EXPECTED_ACTIONS
    ],
    "validation_count": [
        int((y_validation == action).sum())
        for action in EXPECTED_ACTIONS
    ],
    "test_count": [
        int((y_test == action).sum())
        for action in EXPECTED_ACTIONS
    ],
})

display(class_coverage)

missing_training_classes = (
    set(EXPECTED_ACTIONS) - training_classes
)

print("\nClasses Absent from Training:")
print(
    sorted(missing_training_classes)
    if missing_training_classes
    else "None"
)

print(f"\nClasses Learned by the Classifier: {sorted(training_classes)}")

,action,training_count,validation_count,test_count
0,ACTIVATE_PUMP,617,143,104
1,DEACTIVATE_PUMP,551,122,104
2,ACTIVATE_FAN,606,120,132
3,DEACTIVATE_FAN,580,111,128
4,INSPECT_DEVICE,416,89,73
5,NO_ACTION,21410,4563,4595



Classes Absent from Training:
None

Classes Learned by the Classifier: ['ACTIVATE_FAN', 'ACTIVATE_PUMP', 'DEACTIVATE_FAN', 'DEACTIVATE_PUMP', 'INSPECT_DEVICE', 'NO_ACTION']


## Build the Preprocessor

In [23]:
# Identify feature Types from the Training Dataset
categorical_features = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

numeric_features = [
    column
    for column in X_train.columns
    if column not in categorical_features
]

# Numerical Sensor Features
numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median"),
    ),
])

# Categorical Features
categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent"),
    ),
    (
        "encoder",
        OneHotEncoder(handle_unknown="ignore"),
    ),
])

# Apply the Appropriate Transformations to each Feature Group
preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_features,
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_features,
    ),
])

print(f"Numerical Features: {len(numeric_features)}")
print(f"Categorical Features: {len(categorical_features)}")

print("\nCategorical Columns:")
print(categorical_features)

print("\nNumerical Columns:")
print(numeric_features)

Numerical Features: 56
Categorical Features: 2

Categorical Columns:
['device_category', 'zone']

Numerical Columns:
['temperature_c', 'humidity_pct', 'moisture_pct', 'water_ph', 'water_ec_ms_cm', 'water_flow_l_min', 'device_health', 'battery_pct', 'temperature_c_lag_1', 'temperature_c_lag_2', 'temperature_c_lag_4', 'humidity_pct_lag_1', 'humidity_pct_lag_2', 'humidity_pct_lag_4', 'moisture_pct_lag_1', 'moisture_pct_lag_2', 'moisture_pct_lag_4', 'water_ph_lag_1', 'water_ph_lag_2', 'water_ph_lag_4', 'water_ec_ms_cm_lag_1', 'water_ec_ms_cm_lag_2', 'water_ec_ms_cm_lag_4', 'water_flow_l_min_lag_1', 'water_flow_l_min_lag_2', 'water_flow_l_min_lag_4', 'device_health_lag_1', 'device_health_lag_2', 'device_health_lag_4', 'battery_pct_lag_1', 'battery_pct_lag_2', 'battery_pct_lag_4', 'temperature_c_rolling_mean', 'temperature_c_rolling_std', 'humidity_pct_rolling_mean', 'humidity_pct_rolling_std', 'moisture_pct_rolling_mean', 'moisture_pct_rolling_std', 'water_ph_rolling_mean', 'water_ph_rollin

## Create the Model Pipeline

In [24]:
random_forest = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced_subsample",
    random_state=RANDOM_SEED,
    n_jobs=-1,
    min_samples_leaf=2,
)

# Bundle preprocessing and Classification into one Deployable Pipeline
model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", random_forest),
])

print("Random Forest Pipeline Configured")
print(f"Trees: {random_forest.n_estimators}")
print(f"Class Weighting: {random_forest.class_weight}")
print(f"Minimum Leaf Size: {random_forest.min_samples_leaf}")

Random Forest Pipeline Configured
Trees: 300
Class Weighting: balanced_subsample
Minimum Leaf Size: 2


##  Fit the Model

In [25]:
# Fit Preprocessing and Classification using Training
model_pipeline.fit(X_train, y_train)

print("Random Forest Training Completed")

trained_classes = (
    model_pipeline
    .named_steps["classifier"]
    .classes_
)

print("\nClasses Learned by the Model:")
print(trained_classes)

Random Forest Training Completed

Classes Learned by the Model:
['ACTIVATE_FAN' 'ACTIVATE_PUMP' 'DEACTIVATE_FAN' 'DEACTIVATE_PUMP'
 'INSPECT_DEVICE' 'NO_ACTION']


## Compare Against the Baseline

In [26]:
# Generate Predictions using the Validation Partition
validation_predictions = model_pipeline.predict(X_validation)

rf_accuracy = accuracy_score(
    y_validation,
    validation_predictions,
)

rf_macro_f1 = f1_score(
    y_validation,
    validation_predictions,
    average="macro",
    zero_division=0,
)

validation_results = pd.DataFrame([
    {
        "model": "Majority-Class Baseline",
        "accuracy": baseline_accuracy,
        "macro_f1": baseline_macro_f1,
    },
    {
        "model": "Random Forest",
        "accuracy": rf_accuracy,
        "macro_f1": rf_macro_f1,
    },
])

display(
    validation_results.style.format({
        "accuracy": "{:.4f}",
        "macro_f1": "{:.4f}",
    })
)

print(f"Random Forest Accuracy: {rf_accuracy:.4f}")
print(f"Random Forest Macro-F1: {rf_macro_f1:.4f}")

print(
    "\nMacro-F1 improvement over Baseline: "
    f"{rf_macro_f1 - baseline_macro_f1:+.4f}"
)

,model,accuracy,macro_f1
0,Majority-Class Baseline,0.8864,0.1566
1,Random Forest,0.8902,0.3823


Random Forest Accuracy: 0.8902
Random Forest Macro-F1: 0.3823

Macro-F1 improvement over Baseline: +0.2257


# Model Evaluation and Interpretation

## Evaluate on the Test Set

In [27]:
from sklearn.metrics import accuracy_score, f1_score

# Generate Predictions on the Test Data
test_predictions = model_pipeline.predict(X_test)

# Evaluate the majority class baseline on the same Test observations
baseline_test_predictions = baseline_model.predict(X_test)

# Calculate Test Metrics for both models
test_results = pd.DataFrame([
    {
        "model": "Majority-Class Baseline",
        "accuracy": accuracy_score(
            y_test,
            baseline_test_predictions,
        ),
        "macro_f1": f1_score(
            y_test,
            baseline_test_predictions,
            average="macro",
            zero_division=0,
        ),
    },
    {
        "model": "Random Forest",
        "accuracy": accuracy_score(
            y_test,
            test_predictions,
        ),
        "macro_f1": f1_score(
            y_test,
            test_predictions,
            average="macro",
            zero_division=0,
        ),
    },
])

display(
    test_results.style.format({
        "accuracy": "{:.4f}",
        "macro_f1": "{:.4f}",
    })
)

print("Test Evaluation Completed")

,model,accuracy,macro_f1
0,Majority-Class Baseline,0.8947,0.1574
1,Random Forest,0.8842,0.3501


Test Evaluation Completed


## Evaluate Individual Operator Actions

In [28]:
from sklearn.metrics import classification_report

# Report Every intended Action
expected_actions = [
    "ACTIVATE_PUMP",
    "DEACTIVATE_PUMP",
    "ACTIVATE_FAN",
    "DEACTIVATE_FAN",
    "INSPECT_DEVICE",
    "NO_ACTION",
]

print(
    classification_report(
        y_test,
        test_predictions,
        labels=expected_actions,
        zero_division=0,
        digits=4,
    )
)

                 precision    recall  f1-score   support

  ACTIVATE_PUMP     0.0800    0.0385    0.0519       104
DEACTIVATE_PUMP     0.3933    0.5673    0.4646       104
   ACTIVATE_FAN     0.0000    0.0000    0.0000       132
 DEACTIVATE_FAN     0.6000    0.0469    0.0870       128
 INSPECT_DEVICE     0.5714    0.5479    0.5594        73
      NO_ACTION     0.9127    0.9645    0.9379      4595

       accuracy                         0.8842      5136
      macro avg     0.4262    0.3609    0.3501      5136
   weighted avg     0.8492    0.8842    0.8597      5136



## Compare Test and Validation Performance

In [29]:
# Compare the Metrics
performance_comparison = pd.DataFrame([
    {
        "dataset": "Validation",
        "accuracy": rf_accuracy,
        "macro_f1": rf_macro_f1,
    },
    {
        "dataset": "Test",
        "accuracy": accuracy_score(y_test, test_predictions),
        "macro_f1": f1_score(
            y_test,
            test_predictions,
            average="macro",
            zero_division=0,
        ),
    },
])

display(
    performance_comparison.style.format({
        "accuracy": "{:.4f}",
        "macro_f1": "{:.4f}",
    })
)

,dataset,accuracy,macro_f1
0,Validation,0.8902,0.3823
1,Test,0.8842,0.3501
